In [ ]:
import pandas as pd

books_final = pd.read_csv('data/books_final.csv')
print(books_final.shape)
books_final.head()

In [ ]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer

caminho_embeddings = 'data/embeddings.npy'

if os.path.exists(caminho_embeddings):
    print("Embeddings já existem, carregando do disco...")
    embeddings = np.load(caminho_embeddings)
else:
    print("Embeddings não encontrados, gerando...")
    modelo = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = modelo.encode(
        books_final['genero_texto'].tolist(),
        show_progress_bar=True
    )
    np.save(caminho_embeddings, embeddings)
    print("Embeddings gerados e salvos.")

print(embeddings.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similaridade = cosine_similarity(embeddings)
print(similaridade.shape)

In [ ]:
indices = pd.Series(books_final.index, index=books_final['title']).drop_duplicates()

In [ ]:
def recommender(titulo, n=5, rating_minimo=0):
    if titulo not in indices:
        print(f"Livro '{titulo}' não encontrado na base.")
        return None
    
    idx = indices[titulo]
    scores = list(enumerate(similaridade[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:]  # remove o próprio livro
    
    resultado = books_final.iloc[[i[0] for i in scores]].copy()
    resultado['similaridade'] = [i[1] for i in scores]
    
    # filtra por rating mínimo, se especificado
    resultado = resultado[resultado['average_rating'] >= rating_minimo]
    
    return resultado[['title', 'authors', 'genero_texto', 'average_rating', 'similaridade']].head(n)

In [ ]:
recommender('The Final Empire (Mistborn, #1)', 10, 4)